# Data Visualizations

In [1]:
# import packages 
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import glob
import os

In [113]:
# colors 

'#f5f0e8'   # body
'#1a1a1a'   # background
'#06395b'
'#0c629b'
"#458dbd"
"#adddfc"
'#1a5c3a'
"#22794c"
"#67a384"
"#b4d9c6"
"#c48b22"
"#e3b96b"
"#f7dfb3"
"#b84141"

color1 = ['#06395b', "#458dbd", "#adddfc", 'rgba(6, 57, 91, 0.3)', 'rgba(69, 141, 189, 0.3)' ]
color2 = ['#1a5c3a', "#4f916f", "#b4d9c6", 'rgba(79, 145, 111, 0.3)', 'rgba(180, 217, 198, 0.3)' ]
color3 = ["#c48b22", "#e3b96b", "#f7dfb3", 'rgba(196, 139, 34, 0.3)', 'rgba(227, 185, 107, 0.3)' ]

## Overall Percent Spending used on Food

Food personal consumption expenditures / total personal consumption expenditures over time = what percent of spending is used for food over time in general


In [114]:
# import data

food_pce = pd.read_csv("../data/total/DFXARC1M027SBEA.csv", low_memory=False)
pce = pd.read_csv("../data/total/PCE.csv", low_memory=False)

In [115]:
# clean data 

# filter to 1960 and later 
food_pce_clean = food_pce.loc[food_pce['observation_date']>='1960-01-01']
pce_clean = pce.loc[pce['observation_date']>='1960-01-01']

# rename 
food_pce_clean = food_pce_clean.rename(columns={'DFXARC1M027SBEA':'food_pce'})
pce_clean = pce_clean.rename(columns={'PCE':'pce'})

# merge 
percent_food_expend = food_pce_clean.merge(pce_clean, how='outer')

# divide to get percentage
percent_food_expend['percent'] = (percent_food_expend['food_pce']/percent_food_expend['pce']) *100

In [116]:
# line chart of food as a percent of personal consumption expenditures

# data 
y = percent_food_expend["percent"]
x = percent_food_expend["observation_date"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="",
        line=dict(
            color='#67a384',
            width=4
        ),
        fill='tozeroy',
        fillcolor='rgba(103, 163, 132, 0.18)',

        hovertemplate=
        "%{y:.1f}%   "
    )
)

# title and layout 
fig.update_layout(
    width=1000,
    height=600,
    template="plotly_dark",

    title=dict(
        text="Food Expenditure as a Share of Total Personal Consumption Expenditures",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=10
    ),

    yaxis=dict(
        title="Percentage",
        range=[0, 100],
        ticksuffix="%",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    )
)

# endpoint annotation
fig.add_annotation(
    x=x.iloc[-1],
    y=y.iloc[-1],
    text=f"{y.iloc[-1]:.1f}%",
    showarrow=False,
    xshift=20,
    font=dict(
        color="#f7dfb3",
        size=14
    )
)

fig.show()

In [117]:
# save fig 
fig.write_html(f"../website/charts/percent_food_expenditure.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

## Food CPI vs General CPI 

Food cpi vs general cpi  = whether food prices have inflated faster or slower than everything else in the economy over time / is food getting more expensive relative to other things Americans buy?

In [138]:
# import data 

food_cpi = pd.read_csv("../data/total/CPIUFDSL.csv", low_memory=False)
general_cpi = pd.read_csv("../data/total/CPIAUCSL.csv", low_memory=False)

In [139]:
# clean data 

# filter to 1960 and later 
food_cpi_clean = food_cpi.loc[food_cpi['observation_date']>='1960-01-01']
general_cpi_clean = general_cpi.loc[general_cpi['observation_date']>='1960-01-01']

# rename 
food_cpi_clean = food_cpi_clean.rename(columns={'CPIUFDSL':'food_cpi'})
general_cpi_clean = general_cpi_clean.rename(columns={'CPIAUCSL':'total_cpi'})

# merge 
both_cpi = food_cpi_clean.merge(general_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_food = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_cpi"].values
base_all = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "total_cpi"].values

both_cpi['food_cpi_reindex'] = (both_cpi['food_cpi'] / base_food) * 100
both_cpi['total_cpi_reindex'] = (both_cpi['total_cpi'] / base_all) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [140]:
# line chart of food as a percent of personal consumption expenditures

# data 
x = both_cpi["observation_date"]
food_cpi = both_cpi["food_cpi_reindex"]
total_cpi = both_cpi["total_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        hovertemplate=
        "All Items CPI: %{y:.1f}        <extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_cpi,
        mode="lines",
        name="Food",
        line=dict(
            color='#67a384',
            width=4
        ),
        hovertemplate=
        "Food CPI: %{y:.1f}     <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    width=1000, 
    height=600, 

    template="plotly_dark",

    title=dict(
        text="Food vs. Overall Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=80, r=30, t=80, b=60),

    hovermode="x unified",
    hoverlabel=dict(
        align='auto',
        font_size=12
    ),

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        entrywidth=50,
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False,
        font=dict(
            size=12)
    )
)

fig.show()

In [141]:
# save fig 
fig.write_html(f"../website/charts/food_vs_overall_inflation.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

## Food at Home vs Food Away from Home

Food at home vs food away from home CPIs over time = whether eating at home vs eating out was cheaper at certain times 

In [142]:
# import data 

food_home_cpi = pd.read_csv("../data/categories/CUUR0000SAF11.csv", low_memory=False)
food_away_cpi = pd.read_csv("../data/categories/CUUR0000SEFV.csv", low_memory=False)

In [143]:
# clean data 

# filter to 1960 and later 
food_home_cpi_clean = food_home_cpi.loc[food_home_cpi['observation_date']>='1960-01-01']
food_away_cpi_clean = food_away_cpi.loc[food_away_cpi['observation_date']>='1960-01-01']

# rename 
food_home_cpi_clean = food_home_cpi_clean.rename(columns={'CUUR0000SAF11':'food_home_cpi'})
food_away_cpi_clean = food_away_cpi_clean.rename(columns={'CUUR0000SEFV':'food_away_cpi'})

# merge 
both_cpi = food_home_cpi_clean.merge(food_away_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_home = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_home_cpi"].values
base_away = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_away_cpi"].values

both_cpi['food_home_cpi_reindex'] = (both_cpi['food_home_cpi'] / base_home) * 100
both_cpi['food_away_cpi_reindex'] = (both_cpi['food_away_cpi'] / base_away) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [144]:
# line chart of food at home vs food away from home cpis 

# data 
x = both_cpi["observation_date"]
food_home_cpi = both_cpi["food_home_cpi_reindex"]
food_away_cpi = both_cpi["food_away_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Home CPI: %{y:.1f}       <extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_away_cpi,
        mode="lines",
        name="Food Away from Home",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Away CPI: %{y:.1f}       <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    width=1000, 
    height=600, 

    template="plotly_dark",

    title=dict(
        text="Food at Home vs. Away from Home Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=90, r=0, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        entrywidth=140,
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.85,
        itemclick=False,
        itemdoubleclick=False,
        font=dict(
            size=12)
    )
)

fig.show()

In [145]:
# save fig 
fig.write_html(f"../website/charts/home_vs_away.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

## Grocery Category Comparison 

Grocery categories cpi comparison = relative price trajectory of different food types, is cheaper food less healthy now


In [146]:
# import data 

cereal_bakery = pd.read_csv("../data/categories/CUUR0000SAF111.csv", low_memory=False)
meat_fish_eggs = pd.read_csv("../data/categories/CUUR0000SAF112.csv", low_memory=False)
fruit_veg = pd.read_csv("../data/categories/CUUR0000SAF113.csv", low_memory=False)
beverage = pd.read_csv("../data/categories/CUUR0000SAF114.csv", low_memory=False)

# clean data 

# filter to 1960 and later 
cereal_bakery = cereal_bakery.loc[cereal_bakery['observation_date']>='1967-01-01']
meat_fish_eggs = meat_fish_eggs.loc[meat_fish_eggs['observation_date']>='1967-01-01']
fruit_veg = fruit_veg.loc[fruit_veg['observation_date']>='1967-01-01']
beverage = beverage.loc[beverage['observation_date']>='1967-01-01']

# rename 
cereal_bakery = cereal_bakery.rename(columns={'CUUR0000SAF111':'cereal_bakery_cpi'})
meat_fish_eggs = meat_fish_eggs.rename(columns={'CUUR0000SAF112':'meat_fish_eggs_cpi'})
fruit_veg = fruit_veg.rename(columns={'CUUR0000SAF113':'fruit_veg_cpi'})
beverage = beverage.rename(columns={'CUUR0000SAF114':'beverage_cpi'})

# reindex to January 1967 = 100
base_cb = cereal_bakery.loc[cereal_bakery['observation_date']=="1967-01-01", "cereal_bakery_cpi"].values
base_mfe = meat_fish_eggs.loc[meat_fish_eggs['observation_date']=="1967-01-01", "meat_fish_eggs_cpi"].values
base_fv = fruit_veg.loc[fruit_veg['observation_date']=="1967-01-01", "fruit_veg_cpi"].values
base_b = beverage.loc[beverage['observation_date']=="1967-01-01", "beverage_cpi"].values

cereal_bakery['cereal_bakery_cpi_reindex'] = (cereal_bakery['cereal_bakery_cpi'] / base_cb) * 100
meat_fish_eggs['meat_fish_eggs_cpi_reindex'] = (meat_fish_eggs['meat_fish_eggs_cpi'] / base_mfe) * 100
fruit_veg['fruit_veg_cpi_reindex'] = (fruit_veg['fruit_veg_cpi'] / base_fv) * 100
beverage['beverage_cpi_reindex'] = (beverage['beverage_cpi'] / base_b) * 100

# drop na 
cereal_bakery = cereal_bakery.dropna()
meat_fish_eggs = meat_fish_eggs.dropna()
fruit_veg = fruit_veg.dropna()
beverage = beverage.dropna()

In [147]:
# line chart of grocery category cpis 

# data 
x = cereal_bakery["observation_date"]
cb = cereal_bakery["cereal_bakery_cpi_reindex"]
mfe = meat_fish_eggs["meat_fish_eggs_cpi_reindex"]
fv = fruit_veg["fruit_veg_cpi_reindex"]
b = beverage["beverage_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=cb,
        mode="lines",
        name="Cereal and Bakery",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Cereal and Bakery CPI: %{y:.1f}   <extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=mfe,
        mode="lines",
        name="Meat, Poultry, Fish, Eggs",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Meat, Poultry, Fish, Eggs CPI: %{y:.1f}              <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=fv,
        mode="lines",
        name="Fruit and Vegetables",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Fruit and Vegetables CPI: %{y:.1f}   <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=b,
        mode="lines",
        name="Beverages",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Beverages CPI: %{y:.1f}   <extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    width=1000, 
    height=600, 

    template="plotly_dark",

    title=dict(
        text="Grocery Price Inflation by Category",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=80, r=30, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        entrywidth=160,
        yanchor="middle",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(
            size=12)
    )
)

fig.show()

In [148]:
# save fig 
fig.write_html(f"../website/charts/grocery_categories.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

## Animated Bar Chart by Outlet 

Monthly sales by outlet = compare sales by outlet over time in order to see whether more people are food at grocery stores, large stores like costco, restaurants, etc

In [14]:
# import data 

sales_by_outlet = pd.read_csv("../data/usda/monthly_sales_by_outlet.csv", low_memory=False)

In [15]:
# clean data 

# select columns 
sales_by_outlet_clean = sales_by_outlet[['Year',
                                   'Month',
                                   'Food-at-home sales at grocery stores (millions of nominal U.S. dollars)',
                                   'Food-at-home sales at warehouse clubs and supercenters (millions of nominal U.S. dollars)',
                                   'Other food-at-home sales (millions of nominal U.S. dollars)',
                                   'Food sales at full-service restaurants (millions of nominal U.S. dollars)',
                                   'Food sales at limited-service restaurants (millions of nominal U.S. dollars)',
                                   'Other food-away-from-home sales (millions of nominal U.S. dollars)',
                                   'Total food-at-home sales (millions of nominal U.S. dollars)',
                                   'Total food-away-from-home sales (millions of nominal U.S. dollars)',
                                   'Total food sales (millions of nominal U.S. dollars)']]

# rename columns 
sales_by_outlet_clean = sales_by_outlet_clean.rename(columns = {
    'Food-at-home sales at grocery stores (millions of nominal U.S. dollars)': 'grocery_stores',
    'Food-at-home sales at warehouse clubs and supercenters (millions of nominal U.S. dollars)': 'warehouse_supercenter',
    'Other food-at-home sales (millions of nominal U.S. dollars)': 'other_food_at_home',
    'Food sales at full-service restaurants (millions of nominal U.S. dollars)': 'full_restaurants',
    'Food sales at limited-service restaurants (millions of nominal U.S. dollars)': 'limited_restaurants',
    'Other food-away-from-home sales (millions of nominal U.S. dollars)': 'other_food_away',
    'Total food-at-home sales (millions of nominal U.S. dollars)': 'total_food_at_home',
    'Total food-away-from-home sales (millions of nominal U.S. dollars)': 'total_food_away',
    'Total food sales (millions of nominal U.S. dollars)': 'total_overall'
})

# combine year and month into date column 
sales_by_outlet_clean['date'] = pd.to_datetime(sales_by_outlet_clean['Month'] + ' ' + sales_by_outlet_clean['Year'].astype(str), format='%B %Y')
sales_by_outlet_clean['Date '] = sales_by_outlet_clean['date'].dt.strftime('%B %Y')

# convert sales to float 
sales_by_outlet_clean['grocery_stores'] = sales_by_outlet_clean['grocery_stores'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['warehouse_supercenter'] = sales_by_outlet_clean['warehouse_supercenter'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['other_food_at_home'] = sales_by_outlet_clean['other_food_at_home'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['full_restaurants'] = sales_by_outlet_clean['full_restaurants'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['limited_restaurants'] = sales_by_outlet_clean['limited_restaurants'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['other_food_away'] = sales_by_outlet_clean['other_food_away'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['total_food_at_home'] = sales_by_outlet_clean['total_food_at_home'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['total_food_away'] = sales_by_outlet_clean['total_food_away'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['total_overall'] = sales_by_outlet_clean['total_overall'].astype(str).str.replace(',', '', regex=False).astype(float)

# convert to percentage
sales_by_outlet_clean['percent_grocery_stores'] = (sales_by_outlet_clean['grocery_stores']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_warehouse_supercenter'] = (sales_by_outlet_clean['warehouse_supercenter']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_other_food_at_home'] = (sales_by_outlet_clean['other_food_at_home']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_full_restaurants'] = (sales_by_outlet_clean['full_restaurants']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_limited_restaurants'] = (sales_by_outlet_clean['limited_restaurants']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_other_food_away'] = (sales_by_outlet_clean['other_food_away']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_total_food_at_home'] = (sales_by_outlet_clean['total_food_at_home']/sales_by_outlet_clean['total_overall']) *100
sales_by_outlet_clean['percent_total_food_away'] = (sales_by_outlet_clean['total_food_away']/sales_by_outlet_clean['total_overall']) *100


# convert to long format
sales_by_outlet_long = sales_by_outlet_clean.melt(
    id_vars=['Year', 'Month', 'date', 'Date '],
    value_vars=[
        'percent_grocery_stores',
        'percent_warehouse_supercenter',
        'percent_other_food_at_home',
        'percent_full_restaurants',
        'percent_limited_restaurants',
        'percent_other_food_away',
        # 'total_food_at_home',
        # 'total_food_away',
        # 'total_overall'
    ],
    var_name='category',
    value_name='sales'
)


# sort values 
sales_by_outlet_long = sales_by_outlet_long.sort_values(by=['date','category'])

# add columns for hover data 
sales_by_outlet_long['Category '] = sales_by_outlet_long['category'].map({
    'percent_grocery_stores': ' Grocery Stores',
    'percent_warehouse_supercenter': ' Warehouse Clubs',
    'percent_other_food_at_home': ' Other, at Home',
    'percent_full_restaurants': ' Full Restaurants',
    'percent_limited_restaurants': ' Limited Restaurants',
    'percent_other_food_away': ' Other, Away from Home',
})
sales_by_outlet_long['Percentage '] = ' '+sales_by_outlet_long['sales'].round(2).astype(str)+"%"

In [16]:
# make plot 

# colors
color_map = {
    'percent_grocery_stores': "#458dbd",
    'percent_warehouse_supercenter': "#67a384",
    'percent_other_food_at_home': "#adddfc",
    'percent_full_restaurants': "#b84141",
    'percent_limited_restaurants': "#e38a72",
    'percent_other_food_away': "#e3b96b",
}

# plot 
fig = px.bar(sales_by_outlet_long, 
             x="category", 
             y="sales", 
             color="category",
             color_discrete_map=color_map,
             animation_frame="date",
             animation_group="category",
             range_y=[0,100],
             hover_name="Date ",
             hover_data={
                "Category ":True,
                "Percentage ":True,
                "category":False,
                "date":False,
                "sales":False
                },
             )

fig.update_traces(
    marker_line_width=0,
)


fig.update_layout(
    width=1000, 
    height=600, 
    
    # title
    title={
        'text': 'Share of Food Sales by Outlet Type over Time',
        'x': 0.02,
        'xanchor': 'left',
        'font': {
            'size': 24,
            'family': 'Playfair Display',
            'color': '#f5f5f5'
        }
    },

    # plot background
    plot_bgcolor='#1a1a1a',

    # entire figure background
    paper_bgcolor='#1a1a1a',

    # x-axis
    xaxis=dict(
        title='Outlet Category',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=12,
            color='#f5f5f5'
        )
    ),

    # y-axis
    yaxis=dict(
        title='Percentage of Total Food Sales (Monthly)',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=16,
            color='#f5f5f5'
        )
    ),

    # legend
    showlegend=False
)

fig.update_xaxes(
    tickvals=[
        'percent_grocery_stores',
        'percent_warehouse_supercenter',
        'percent_other_food_at_home',
        'percent_full_restaurants',
        'percent_limited_restaurants',
        'percent_other_food_away'
    ],
    ticktext=[
        'Grocery Stores',
        'Warehouse Clubs',
        'Other, at Home',
        'Full Restaurants',
        'Limited Restaurants',
        'Other, Away from Home'
    ]
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zerolinecolor="rgba(255,255,255,0.08)",
    tickfont=dict(size=12),
    ticksuffix="%"
)

for step in fig.layout.sliders[0].steps:
    step["label"] = step["label"][:7]

fig.update_layout(

    # animation slider
    sliders=[{
        'currentvalue': {
            'font': {'color': '#f5f5f5'},
            'prefix': 'Date: ',
            'visible': True,
        },

        # tick labels on slider
        'tickcolor': "rgba(245, 245, 245, 0.2)",

        # slider styling
        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",
        'borderwidth': 1,

        # font for slider step labels
        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        }
    }],

    # buttons
    updatemenus=[{
        'type': 'buttons',

        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        },

        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",

    }]
)

fig.show()

In [19]:
# save fig 
fig.write_html(f"../website/charts/spending_by_outlet.html", include_plotlyjs=True, full_html=True, config={'responsive': False}, auto_play=False)

In [171]:
sales_by_outlet_clean.head()

,Year,Month,grocery_stores,warehouse_supercenter,other_food_at_home,full_restaurants,limited_restaurants,other_food_away,total_food_at_home,total_food_away,...,date,Date,percent_grocery_stores,percent_warehouse_supercenter,percent_other_food_at_home,percent_full_restaurants,percent_limited_restaurants,percent_other_food_away,percent_total_food_at_home,percent_total_food_away
0,2024,December,53508.70,27638.43,17470.23,49109.64,44980.43,22796.67,98617.36,116886.74,...,2024-12-01,December 2024,24.829551,12.825014,8.106681,22.788262,20.872192,10.578300,45.761245,54.238755
1,2024,November,51907.11,24328.66,15512.88,46314.84,44311.91,22347.16,91748.65,112973.91,...,2024-11-01,November 2024,25.354856,11.883722,7.577514,22.623222,21.644859,10.915827,44.816092,55.183908
2,2024,October,51349.68,23279.29,15250.78,46335.73,46825.90,22793.14,89879.75,115954.77,...,2024-10-01,October 2024,24.947069,11.309711,7.409243,22.511156,22.749294,11.073526,43.666024,56.333976
3,2024,September,49193.49,21906.63,13987.02,44877.04,44574.77,22187.63,85087.14,111639.44,...,2024-09-01,September 2024,25.006021,11.135572,7.109878,22.811884,22.658235,11.278410,43.251471,56.748529
4,2024,August,51375.86,24113.02,14548.53,48088.59,48359.83,23034.21,90037.41,119482.63,...,2024-08-01,August 2024,24.520738,11.508694,6.943742,22.951785,23.081243,10.993798,42.973173,57.026827


In [176]:
display(sales_by_outlet_clean.loc[sales_by_outlet_clean['percent_total_food_at_home'].idxmin()])
display(sales_by_outlet_clean.loc[sales_by_outlet_clean['percent_total_food_at_home'].idxmax()])
display(sales_by_outlet_clean.loc[sales_by_outlet_clean['percent_total_food_away'].idxmin()])
display(sales_by_outlet_clean.loc[sales_by_outlet_clean['percent_total_food_away'].idxmax()])

Year                                            2024
Month                                          April
grocery_stores                              47487.91
warehouse_supercenter                       21438.86
other_food_at_home                          14659.33
full_restaurants                            44849.51
limited_restaurants                         46274.21
other_food_away                             22415.77
total_food_at_home                           83586.1
total_food_away                            113539.49
total_overall                              197125.59
date                             2024-04-01 00:00:00
Date                                      April 2024
percent_grocery_stores                      24.09018
percent_warehouse_supercenter              10.875737
percent_other_food_at_home                  7.436543
percent_full_restaurants                   22.751744
percent_limited_restaurants                23.474481
percent_other_food_away                    11.

Year                                            2020
Month                                          April
grocery_stores                              42658.66
warehouse_supercenter                       16638.27
other_food_at_home                          11306.53
full_restaurants                             8879.52
limited_restaurants                         21248.28
other_food_away                             11190.68
total_food_at_home                          70603.46
total_food_away                             41318.48
total_overall                              111921.94
date                             2020-04-01 00:00:00
Date                                      April 2020
percent_grocery_stores                     38.114654
percent_warehouse_supercenter              14.865959
percent_other_food_at_home                 10.102157
percent_full_restaurants                    7.933672
percent_limited_restaurants                 18.98491
percent_other_food_away                     9.

Year                                            2020
Month                                          April
grocery_stores                              42658.66
warehouse_supercenter                       16638.27
other_food_at_home                          11306.53
full_restaurants                             8879.52
limited_restaurants                         21248.28
other_food_away                             11190.68
total_food_at_home                          70603.46
total_food_away                             41318.48
total_overall                              111921.94
date                             2020-04-01 00:00:00
Date                                      April 2020
percent_grocery_stores                     38.114654
percent_warehouse_supercenter              14.865959
percent_other_food_at_home                 10.102157
percent_full_restaurants                    7.933672
percent_limited_restaurants                 18.98491
percent_other_food_away                     9.

Year                                            2024
Month                                          April
grocery_stores                              47487.91
warehouse_supercenter                       21438.86
other_food_at_home                          14659.33
full_restaurants                            44849.51
limited_restaurants                         46274.21
other_food_away                             22415.77
total_food_at_home                           83586.1
total_food_away                            113539.49
total_overall                              197125.59
date                             2024-04-01 00:00:00
Date                                      April 2024
percent_grocery_stores                      24.09018
percent_warehouse_supercenter              10.875737
percent_other_food_at_home                  7.436543
percent_full_restaurants                   22.751744
percent_limited_restaurants                23.474481
percent_other_food_away                    11.

In [172]:
sales_by_outlet_clean.tail()

,Year,Month,grocery_stores,warehouse_supercenter,other_food_at_home,full_restaurants,limited_restaurants,other_food_away,total_food_at_home,total_food_away,...,date,Date,percent_grocery_stores,percent_warehouse_supercenter,percent_other_food_at_home,percent_full_restaurants,percent_limited_restaurants,percent_other_food_away,percent_total_food_at_home,percent_total_food_away
331,1997,May,23432.74,2149.88,5610.42,10509.64,10112.04,4447.09,31193.04,25068.77,...,1997-05-01,May 1997,41.649460,3.821207,9.971986,18.679882,17.973186,7.904278,55.442653,44.557347
332,1997,April,21509.06,1914.52,5403.54,9831.41,9332.29,4205.16,28827.12,23368.86,...,1997-04-01,April 1997,41.208269,3.667945,10.352406,18.835569,17.879327,8.056483,55.228621,44.771379
333,1997,March,22756.94,1969.54,5613.02,10093.29,9340.53,4159.77,30339.50,23593.59,...,1997-03-01,March 1997,42.194764,3.651821,10.407377,18.714466,17.318737,7.712835,56.253962,43.746038
334,1997,February,20104.82,1694.26,5246.76,9094.30,8255.88,3922.10,27045.84,21272.28,...,1997-02-01,February 1997,41.609276,3.506469,10.858783,18.821717,17.086509,8.117245,55.974529,44.025471
335,1997,January,22020.99,1747.02,5372.48,9153.22,8302.23,3960.81,29140.49,21416.26,...,1997-01-01,January 1997,43.556973,3.455562,10.626632,18.104843,16.421605,7.834384,57.639168,42.360832


## Inequality 1984 vs 2024

DOUBLE BAR CHART w/ 1984 vs 2024 for all 3 percentiles 
LINE CHART for total percentage over time and percentiles

Food at home expenditures vs income, compare across percentiles over time 

pre-tax income 
- slightly understates food spend percentage for lower-income households bc income is higher than what they actually have 
- undercounts their actual resources (SNAP benefits, earned income tax credit, etc)
- the most consistently reported figure

In [149]:
# import data 

all_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0101M.csv", low_memory=False)
bottom_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0102M.csv", low_memory=False)
middle_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0104M.csv", low_memory=False)
top_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0106M.csv", low_memory=False)
all_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0101M.csv", low_memory=False)
bottom_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0102M.csv", low_memory=False)
middle_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0104M.csv", low_memory=False)
top_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0106M.csv", low_memory=False)

# clean data 

# rename 
all_food_spend = all_food_spend.rename(columns={'CXUFOODHOMELB0101M':'all_food_spend'})
bottom_food_spend = bottom_food_spend.rename(columns={'CXUFOODHOMELB0102M':'bottom_food_spend'})
middle_food_spend = middle_food_spend.rename(columns={'CXUFOODHOMELB0104M':'middle_food_spend'})
top_food_spend = top_food_spend.rename(columns={'CXUFOODHOMELB0106M':'top_food_spend'})
all_income = all_income.rename(columns={'CXUINCBEFTXLB0101M':'all_income'})
bottom_income = bottom_income.rename(columns={'CXUINCBEFTXLB0102M':'bottom_income'})
middle_income = middle_income.rename(columns={'CXUINCBEFTXLB0104M':'middle_income'})
top_income = top_income.rename(columns={'CXUINCBEFTXLB0106M':'top_income'})

# drop na 
all_food_spend = all_food_spend.dropna()
bottom_food_spend = bottom_food_spend.dropna()
middle_food_spend = middle_food_spend.dropna()
top_food_spend = top_food_spend.dropna()
all_income = all_income.dropna()
bottom_income = bottom_income.dropna()
middle_income = middle_income.dropna()
top_income = top_income.dropna()

In [150]:
## line chart of food spending percentage by income percentile 

# data 
x = all_food_spend["observation_date"]
overall = (all_food_spend["all_food_spend"]/all_income['all_income']) *100
bottom = (bottom_food_spend["bottom_food_spend"]/bottom_income['bottom_income']) *100
middle = (middle_food_spend["middle_food_spend"]/middle_income['middle_income']) *100
top = (top_food_spend["top_food_spend"]/top_income['top_income']) *100

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=overall,
        mode="lines",
        name="Overall",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Overall: %{y:.2f}%     <extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=bottom,
        mode="lines",
        name="Bottom 20%",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Bottom 20%: %{y:.1f}%         <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=middle,
        mode="lines",
        name="Middle 20%",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Middle 20%: %{y:.1f}%     <extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=top,
        mode="lines",
        name="Top 20%",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Top 20%: %{y:.1f}%     <extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    width=1000, 
    height=600, 
    template="plotly_dark",

    title=dict(
        text="Share of Income Spent on Groceries Over Time",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=80, r=20, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Food at Home as % of Income Before Taxes",
        tickfont=dict(size=14),
        ticksuffix="%",
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        entrywidth=0.15,
        entrywidthmode='fraction',
        yanchor="top",
        y=1,
        xanchor="right",
        x=1,
        font=dict(
            size=12)
    )
)

fig.show()

In [151]:
# save fig 
fig.write_html(f"../website/charts/line_grocery_income.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

In [26]:
## compare food spending between 2 years across incomes 

years_to_compare = [1984, 2024]
quintiles = ["Bottom 20%", "Middle 20%", "Top 20%"]

# get values for each category 
ratios = {"Bottom 20%": bottom,
          "Middle 20%": middle,
          "Top 20%": top}
values_1984 = [ratios[q][0] for q in quintiles]
values_2024 = [ratios[q].iloc[-1] for q in quintiles]

# plot
fig = go.Figure()

fig.add_trace(go.Bar(
    name="1984",
    x=quintiles,
    y=values_1984,
    marker_color="#0c629b",
    hovertemplate="1984<br>%{x}<br>%{y}<extra></extra>"
))

fig.add_trace(go.Bar(
    name="2024",
    x=quintiles,
    y=values_2024,
    marker_color='#22794c',
    hovertemplate="2024<br>%{x}<br>%{y}<extra></extra>"
))

fig.update_layout(
    width=1000, 
    height=600,
    template="plotly_dark",

    title=dict(
        text="Share of Income Spent on Groceries by Income Group",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=80, r=20, t=80, b=60),

    barmode="group",
    xaxis=dict(
        title="Income Quintile",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Food at Home as % of Income Before Taxes",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False,
        ticksuffix="%",
        range=[0, max(values_1984 + values_2024) * 1.2]
    ),
   
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02, 
        xanchor="right", 
        x=1
    ),

    bargap=0.1,
    bargroupgap=0
)

fig.show()

In [27]:
# save fig 
fig.write_html(f"../website/charts/bar_grocery_income.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False})

## Per State 

State sales per capita (udsa) / per capita personal income per state = food burden by state over time

In [2]:
# import data 

state_food = pd.read_csv("../data/usda/state_sales_per_capita_no_taxes_tips.csv", low_memory=False)

folder_path = "../data/states"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state_abbr'] = file[15:17]
    col_name = file[15:21]
    df = df.rename(columns={col_name:'per_capita_personal_income'})
    dfs.append(df)

state_personal_income = pd.concat(
    dfs,
    ignore_index=True
)

# clean data 

# state mapping
state_map = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
    "DC": "District of Columbia"
}

# filter to 1997 and later
state_personal_income = state_personal_income.loc[state_personal_income['observation_date']>='1997-01-01']

# add year and state cols
state_personal_income['year'] = pd.to_datetime(state_personal_income['observation_date']).dt.year
state_personal_income['state'] = state_personal_income["state_abbr"].map(state_map)

# select columns 
state_food = state_food[['Year', 'State',
       'Food-at-home per capita sales without taxes and tips (nominal U.S. dollars)',
       'Food-away-from-home per capita sales without taxes and tips (nominal U.S. dollars)',
       'Total food per capita sales without taxes and tips (nominal U.S. dollars)']]

# rename columns 
state_food = state_food.rename(columns = {
    'Year':'year', 
    'State':'state',
    'Food-at-home per capita sales without taxes and tips (nominal U.S. dollars)':'food_home_pc_sales',
    'Food-away-from-home per capita sales without taxes and tips (nominal U.S. dollars)':'food_away_pc_sales',
    'Total food per capita sales without taxes and tips (nominal U.S. dollars)':'total_food_pc_sales'
})

# convert sales to float 
state_food['food_home_pc_sales'] = state_food['food_home_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)
state_food['food_away_pc_sales'] = state_food['food_away_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)
state_food['total_food_pc_sales'] = state_food['total_food_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)

# sort values 
state_food = state_food.sort_values(by=['state', 'year'])

# merge
state_food_vs_income = state_food.merge(state_personal_income, on=['year','state'], how='inner')

# add percentage columns 
state_food_vs_income['total_food_income_percentage'] = (state_food_vs_income['total_food_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100
state_food_vs_income['home_food_income_percentage'] = (state_food_vs_income['food_home_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100
state_food_vs_income['away_food_income_percentage'] = (state_food_vs_income['food_away_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100

# add columns for clean hover data 
state_food_vs_income['Year '] = ' '+state_food_vs_income['year'].astype(str)
state_food_vs_income['Percentage '] = ' '+state_food_vs_income['total_food_income_percentage'].round(2).astype(str) + "%"

In [168]:
# color
color_continuous_scale=['#1a5c3a',"#67a384", "#c48b22"]

# plot
fig = px.choropleth(
    state_food_vs_income,
    locations="state_abbr",
    locationmode="USA-states",
    color="total_food_income_percentage",
    hover_name="state",
    hover_data={
        "total_food_income_percentage": False,
        "year": False,
        'state_abbr':False,
        'Year ':True, 
        'Percentage ': True
    },
    animation_frame="year",
    scope="usa",
    color_continuous_scale=color_continuous_scale,
    range_color=(
        state_food_vs_income["total_food_income_percentage"].min(),
        state_food_vs_income["total_food_income_percentage"].max()
    ),
)

fig.update_layout(
    width=1000, 
    height=600, 

    # title
    title={
        'text': "Food Sales as % of Personal Income by State Over Time (Per Capita)",
        'x': 0.02,
        'xanchor': 'left',
        'font': {
            'size': 24,
            'family': 'Playfair Display',
            'color': '#f5f0e8'
        }
    },

    # plot background
    plot_bgcolor='#1a1a1a',

    # entire figure background
    paper_bgcolor='#1a1a1a',

)

# map background
fig.update_geos(
    bgcolor='#1a1a1a',
    lakecolor='#f5f0e8',
)
fig.update_traces(
    marker_line_color="rgba(245, 245, 245, 0.5)",
    marker_line_width=0.5
)

# animation styling 
fig.update_layout(

    # animation slider
    sliders=[{
        'currentvalue': {
            'font': {'color': '#f5f0e8'},
            'prefix': 'Year: ',
            'visible': True,
        },

        # tick labels on slider
        'tickcolor': "rgba(245, 245, 245, 0.2)",

        # slider styling
        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",
        'borderwidth': 1,

        # font for slider step labels
        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        }
    }],

    # buttons
    updatemenus=[{
        'type': 'buttons',

        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        },

        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",

    }]
)

fig.update_coloraxes(
    colorbar_title="Percentage",
    colorbar_ticksuffix="%",
    colorbar_tickfont=dict(
        color="#f5f0e8",
        size=12
    ),
    colorbar_title_font=dict(
        color="#f5f0e8",
        size=14
    )
)

fig.show()

In [169]:
# save fig 
fig.write_html(f"../website/charts/map_food_sales_income.html", include_plotlyjs="cdn", full_html=False, config={'responsive': False}, auto_play=False)

In [4]:
state_food_vs_income.columns

Index(['year', 'state', 'food_home_pc_sales', 'food_away_pc_sales',
       'total_food_pc_sales', 'observation_date', 'per_capita_personal_income',
       'state_abbr', 'total_food_income_percentage',
       'home_food_income_percentage', 'away_food_income_percentage', 'Year ',
       'Percentage '],
      dtype='str')

In [5]:
# stats

display(state_food_vs_income.loc[state_food_vs_income['home_food_income_percentage'].idxmin()])
display(state_food_vs_income.loc[state_food_vs_income['home_food_income_percentage'].idxmax()])
display(state_food_vs_income.loc[state_food_vs_income['away_food_income_percentage'].idxmin()])
display(state_food_vs_income.loc[state_food_vs_income['away_food_income_percentage'].idxmax()])

year                                            2007
state                           District of Columbia
food_home_pc_sales                           1245.56
food_away_pc_sales                           4124.89
total_food_pc_sales                          5370.45
observation_date                          2007-01-01
per_capita_personal_income                     59358
state_abbr                                        DC
total_food_income_percentage                9.047559
home_food_income_percentage                 2.098386
away_food_income_percentage                 6.949173
Year                                            2007
Percentage                                     9.05%
Name: 234, dtype: object

year                                  2017
state                           Washington
food_home_pc_sales                 4812.85
food_away_pc_sales                 2723.59
total_food_pc_sales                7536.44
observation_date                2017-01-01
per_capita_personal_income           56800
state_abbr                              WA
total_food_income_percentage      13.26838
home_food_income_percentage       8.473327
away_food_income_percentage       4.795053
Year                                  2017
Percentage                          13.27%
Name: 1336, dtype: object

year                                   2001
state                           Connecticut
food_home_pc_sales                  1714.34
food_away_pc_sales                  1204.48
total_food_pc_sales                 2918.82
observation_date                 2001-01-01
per_capita_personal_income            45032
state_abbr                               CT
total_food_income_percentage       6.481657
home_food_income_percentage        3.806937
away_food_income_percentage         2.67472
Year                                   2001
Percentage                            6.48%
Name: 172, dtype: object

year                                  2024
state                               Hawaii
food_home_pc_sales                 3178.96
food_away_pc_sales                 6427.98
total_food_pc_sales                9606.94
observation_date                2024-01-01
per_capita_personal_income           69520
state_abbr                              HI
total_food_income_percentage     13.818959
home_food_income_percentage       4.572727
away_food_income_percentage       9.246231
Year                                  2024
Percentage                          13.82%
Name: 335, dtype: object

In [9]:
# all 
display(state_food_vs_income.loc[state_food_vs_income['state']=='Nevada'])

# home avg percent 
print(state_food_vs_income.loc[state_food_vs_income['state']=='Nevada']['home_food_income_percentage'].mean())

# away avg percent 
print(state_food_vs_income.loc[state_food_vs_income['state']=='Nevada']['away_food_income_percentage'].mean())

,year,state,food_home_pc_sales,food_away_pc_sales,total_food_pc_sales,observation_date,per_capita_personal_income,state_abbr,total_food_income_percentage,home_food_income_percentage,away_food_income_percentage,Year,Percentage
784,1997,Nevada,1523.93,2193.16,3717.09,1997-01-01,27464,NV,13.534409,5.548828,7.985581,1997,13.53%
785,1998,Nevada,1550.94,2244.72,3795.66,1998-01-01,29256,NV,12.973954,5.301272,7.672683,1998,12.97%
786,1999,Nevada,1619.06,2486.80,4105.86,1999-01-01,30301,NV,13.550246,5.343256,8.206990,1999,13.55%
787,2000,Nevada,1577.15,2507.63,4084.78,2000-01-01,31858,NV,12.821834,4.950562,7.871273,2000,12.82%
788,2001,Nevada,1654.63,2450.60,4105.23,2001-01-01,32418,NV,12.663428,5.104047,7.559381,2001,12.66%
789,2002,Nevada,1683.90,2418.82,4102.72,2002-01-01,32180,NV,12.749285,5.232753,7.516532,2002,12.75%
790,2003,Nevada,1728.43,2453.53,4181.96,2003-01-01,33136,NV,12.620594,5.216170,7.404424,2003,12.62%
791,2004,Nevada,1808.31,2562.83,4371.14,2004-01-01,35173,NV,12.427544,5.141188,7.286356,2004,12.43%
792,2005,Nevada,1918.82,2654.62,4573.44,2005-01-01,37907,NV,12.064896,5.061915,7.002981,2005,12.06%
793,2006,Nevada,2056.58,2742.68,4799.26,2006-01-01,39437,NV,12.169435,5.214849,6.954586,2006,12.17%


5.330902473387048
7.766675381218611


In [7]:
state_food_vs_income['home_food_income_percentage'].describe()

count    1428.000000
mean        4.966474
std         0.904331
min         2.098386
25%         4.444327
50%         4.978074
75%         5.548551
max         8.473327
Name: home_food_income_percentage, dtype: float64

In [8]:
state_food_vs_income['away_food_income_percentage'].describe()

count    1428.000000
mean        4.531283
std         0.982571
min         2.674720
25%         3.938720
50%         4.374799
75%         4.843898
max         9.246231
Name: away_food_income_percentage, dtype: float64

In [13]:
state_food_vs_income.groupby('state')[['home_food_income_percentage','away_food_income_percentage','total_food_income_percentage']].mean().sort_values('home_food_income_percentage', ascending=False)

,home_food_income_percentage,away_food_income_percentage,total_food_income_percentage
state,,,
Maine,6.653950,4.780746,11.434696
Utah,6.252668,4.679744,10.932413
Washington,6.200787,3.997547,10.198334
Montana,6.138414,4.779860,10.918275
Idaho,6.069320,4.222343,10.291663
New Hampshire,6.019770,4.133677,10.153447
Arizona,5.889829,4.785254,10.675084
Oregon,5.796426,4.691587,10.488013
Kentucky,5.718447,4.957244,10.675691
